# Lab 09 - Vertrauenswürdigkeit: Grounding, Confidence, "Weiß nicht" (Teil IX)

Vertrauen heißt messen. Sie quantifizieren, wie gut Antworten belegt sind
(Citation-Precision und -Recall), leiten aus dem Retrieval ein Confidence-Signal
ab, bauen ein High-Confidence-Gate und bewerten die "Weiß nicht"-Strategie über
Refusal-Precision und -Recall. Drei Fälle zählen: korrekt beantwortet, korrekt
abgelehnt und der gefährliche Fall, in dem ohne Beleg geantwortet wird.

## 1. Setup

Wir nutzen den Dense-Retriever, weil seine Cosine-Scores ein direkt lesbares
Confidence-Signal liefern (0 bis 1). Der Hybrid-Index aus Teil VI bliebe für
die Antwortqualität besser, aber RRF-Scores sind als Confidence schlecht
interpretierbar.

In [ ]:
import re

from common import llm
from common.corpus import load_corpus
from common.goldset import gold_doc_ids
from common import retrieval as R

docs = load_corpus()
dense = R.DenseRetriever(R.chunk_corpus(docs))

GROUNDED = (
    "Sie sind ein technischer Wissensassistent. Antworten Sie nur aus dem Kontext "
    "und belegen Sie jede Aussage mit [doc_id]. Steht es nicht im Kontext, "
    "antworten Sie genau: 'Das geht aus den Unterlagen nicht hervor.'"
)
CITE_RE = re.compile(r"\[\s*(?:doc_id\s*[:=]\s*)?([a-z0-9][a-z0-9_\-]*)\s*\]")
REFUSAL = "nicht hervor"

## 2. Pipeline mit Confidence-Gate

Das Gate prüft die Cosine-Similarity des besten Chunks. Liegt sie unter der
Schwelle, lehnt das System ab, bevor das Modell überhaupt antwortet. So wird
ein ehrliches "Weiß nicht" erzwungen, statt es zu erhoffen.

In [ ]:
def antwort_mit_gate(frage: str, tau: float = 0.45, k: int = 4) -> dict:
    treffer = dense.search(frage, k=k)
    confidence = treffer[0][1] if treffer else 0.0
    if confidence < tau:
        return {"antwort": "Das geht aus den Unterlagen nicht hervor.",
                "confidence": confidence, "gated": True,
                "zitiert": set(), "quellen": [c.doc_id for c, _ in treffer]}
    kontext = R.format_context(treffer)
    antwort = llm.complete(f"Kontext:\n{kontext}\n\nFrage: {frage}\n\nAntwort:",
                           system=GROUNDED, temperature=0.0, max_tokens=512)
    return {"antwort": antwort, "confidence": confidence, "gated": False,
            "zitiert": set(CITE_RE.findall(antwort)),
            "quellen": [c.doc_id for c, _ in treffer]}

## 3. Citation-Precision und -Recall

Gegen die erwarteten Belege aus dem Gold-Set (Dokumente mit Relevanzgrad >= 2).
Precision: wie viele zitierte Quellen sind wirklich relevant? Recall: wie viele
der erwarteten Belege wurden zitiert?

In [ ]:
answerable = ["q01", "q02", "q04", "q05", "q06", "q10"]
fragen = {
    "q01": "Wie hoch ist der zulässige Dauerbetriebsdruck?",
    "q02": "Welches Hydrauliköl soll verwendet werden?",
    "q04": "Mit welchem Anzugsmoment werden die Pumpenkopf-Schrauben angezogen?",
    "q05": "Was bedeutet der Fehlercode E02?",
    "q06": "Welche Schutzausrüstung ist bei der Wartung nötig?",
    "q10": "Welcher Spitzendruck ist kurzzeitig zulässig?",
}

zeilen = []
for qid in answerable:
    r = antwort_mit_gate(fragen[qid])
    gold = gold_doc_ids(qid, min_grade=2)
    zitiert = r["zitiert"]
    tp = len(zitiert & gold)
    prec = tp / len(zitiert) if zitiert else 0.0
    rec = tp / len(gold) if gold else 0.0
    zeilen.append({"qid": qid, "conf": round(r["confidence"], 2),
                   "zitiert": sorted(zitiert), "gold": sorted(gold),
                   "cit_prec": round(prec, 2), "cit_rec": round(rec, 2)})

import pandas as pd

df = pd.DataFrame(zeilen).set_index("qid")
print(df)
print(f"\nMittel  Citation-Precision={df['cit_prec'].mean():.2f}  "
      f"Citation-Recall={df['cit_rec'].mean():.2f}")

## 4. Refusal-Precision und -Recall

Über einen gemischten Satz aus beantwortbaren und nicht beantwortbaren Fragen.
Eine Ablehnung ist korrekt, wenn die Frage wirklich unbeantwortbar war.

In [ ]:
unbeantwortbar = [
    "Wie hoch ist der Schalldruckpegel der Pumpe im Betrieb?",
    "Welche Software-Version hat die Steuerung?",
    "Welche Farbe hat das Pumpengehäuse?",
]

eval_set = [(f, True) for f in fragen.values()] + [(f, False) for f in unbeantwortbar]
tp = fp = fn = tn = 0
for frage, beantwortbar in eval_set:
    r = antwort_mit_gate(frage)
    abgelehnt = r["gated"] or REFUSAL in r["antwort"].lower()
    if not beantwortbar and abgelehnt:
        tn += 1
    elif not beantwortbar and not abgelehnt:
        fp += 1          # gefährlich: ohne Beleg geantwortet
    elif beantwortbar and abgelehnt:
        fn += 1          # zu vorsichtig
    else:
        tp += 1

ref_prec = tn / (tn + fn) if (tn + fn) else 0.0    # von den Ablehnungen korrekt
ref_rec = tn / (tn + fp) if (tn + fp) else 0.0     # von den Unbeantwortbaren abgelehnt
print(f"beantwortet & korrekt: {tp}   fälschlich beantwortet (gefährlich): {fp}")
print(f"korrekt abgelehnt:     {tn}   zu vorsichtig abgelehnt:            {fn}")
print(f"\nRefusal-Precision={ref_prec:.2f}  Refusal-Recall={ref_rec:.2f}")

## Aufgabe 1: Die Schwelle tau einstellen

Variieren Sie tau (0.30, 0.45, 0.60) und beobachten Sie Refusal-Precision und
-Recall. Eine hohe Schwelle lehnt mehr ab (sicherer, aber vorsichtiger), eine
niedrige beantwortet mehr (hilfreicher, aber riskanter). Wo liegt für einen
sicherheitsrelevanten Assistenten das richtige Ende?

In [ ]:
# Ihre Lösung:

### Lösung

In [ ]:
for tau in (0.30, 0.45, 0.60):
    tp = fp = fn = tn = 0
    for frage, beantwortbar in eval_set:
        r = antwort_mit_gate(frage, tau=tau)
        abgelehnt = r["gated"] or REFUSAL in r["antwort"].lower()
        tn += (not beantwortbar and abgelehnt)
        fp += (not beantwortbar and not abgelehnt)
        fn += (beantwortbar and abgelehnt)
        tp += (beantwortbar and not abgelehnt)
    rp = tn / (tn + fn) if (tn + fn) else 0.0
    rr = tn / (tn + fp) if (tn + fp) else 0.0
    print(f"tau={tau:.2f}: korrekt beantwortet={tp} gefährlich={fp} "
          f"Refusal-Precision={rp:.2f} Refusal-Recall={rr:.2f}")

## Aufgabe 2: Den gefährlichen Fall jagen

Suchen Sie eine nicht beantwortbare Frage, die das Gate *durchlässt* (hohe
Cosine-Similarity zu einem thematisch nahen, aber nicht antwortenden Chunk).
Das ist der Fall, gegen den Confidence allein nicht reicht. Was braucht es
zusätzlich? (Vorschau: belegpflichtige Generierung und Output-Prüfung.)

In [ ]:
# Ihre Lösung:

### Lösung

In [ ]:
kandidaten = [
    "Welcher Berstdruck-Sicherheitsfaktor ist zertifiziert?",
    "Welche Wartungsfirma ist für die Pumpe zuständig?",
    "Welcher Druck gilt für die Baureihe P-20?",
]
for f in kandidaten:
    r = antwort_mit_gate(f)
    print(f"conf={r['confidence']:.2f} gated={r['gated']}  {f}")
    if not r["gated"]:
        print("   -> durchgelassen, Antwort:", r["antwort"][:90].replace("\n", " "))
print("\nConfidence misst Ähnlichkeit, nicht Beantwortbarkeit. Die belegpflichtige "
      "Generierung (Refusal im Prompt) fängt einen Teil dieser Fälle zusätzlich ab.")

## Diskussion

- Citation-Precision und -Recall ziehen in unterschiedliche Richtungen. Welche
  ist für einen Wartungsassistenten wichtiger?
- Das Gate und der Prompt-Refusal sind zwei Schichten für dasselbe Ziel. Warum
  sind zwei unabhängige Schichten besser als eine perfekt eingestellte?
- Welche "Weiß nicht"-Schwelle würden Sie in Ihrem System vor den ersten
  Produktivnutzern setzen, und wie würden Sie sie nachjustieren?